In [0]:
storage_account = "demodatalakemarkets"

sas_token = dbutils.secrets.get(scope="fx-pipeline", key="adls-sas-token")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "SAS")
spark.conf.set(f"fs.azure.sas.token.provider.type.{storage_account}.dfs.core.windows.net","org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider")
spark.conf.set(f"fs.azure.sas.fixed.token.{storage_account}.dfs.core.windows.net", sas_token)

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import (StructType, StructField, StringType,DoubleType, LongType)

BRONZE_PATH = f"abfss://datalake@{storage_account}.dfs.core.windows.net/bronze/yfinance/"

schema_bronze = StructType([
    StructField("Date",             StringType(), True),  
    StructField("Open",             DoubleType(), True),
    StructField("High",             DoubleType(), True),
    StructField("Low",              DoubleType(), True),
    StructField("Close",            DoubleType(), True),
    StructField("Volume",           LongType(),   True),
    StructField("_ticker",          StringType(), True),  
    StructField("_ingested_at_utc", StringType(), True),   
])

FX_TICKERS = ["EURUSD=X", "USDPLN=X"]

In [0]:
df_bronze = (spark.read
             .schema(schema_bronze)
             .option("multiLine", "true")
             .json(BRONZE_PATH + "*/ingest_date=*/data.json"))

display(df_bronze.limit(5))

In [0]:
df_clean = df_bronze.select(
    F.to_date("Date").alias("date"),
    F.col("_ticker").alias("ticker"),
    F.col("Open").alias("open"),
    F.col("High").alias("high"),
    F.col("Low").alias("low"),
    F.col("Close").alias("close"),
    F.col("Volume").alias("volume"),
    F.to_timestamp("_ingested_at_utc").alias("ingested_at"),
)

display(df_clean.limit(5))

In [0]:
w_dedup = Window.partitionBy("ticker", "date").orderBy(F.col("ingested_at").desc())
df_dedup = (df_clean
            .withColumn("rn", F.row_number().over(w_dedup))
            .filter(F.col("rn") == 1)  
            .drop("rn"))

In [0]:
fx  = df_dedup.filter(F.col("ticker").isin(FX_TICKERS))
stk = df_dedup.filter(~F.col("ticker").isin(FX_TICKERS))

kalendarz_gieldy = stk.select("date").distinct()

os_dat = kalendarz_gieldy.unionByName(fx.select("date")).distinct()

In [0]:
fx_tickers = fx.select("ticker").distinct()
szkielet = os_dat.crossJoin(F.broadcast(fx_tickers))

fx_left = szkielet.join(F.broadcast(fx), on=["date", "ticker"], how="left")

w_ffill = (Window.partitionBy("ticker")
           .orderBy("date")
           .rowsBetween(Window.unboundedPreceding, Window.currentRow))

fx_filled = (fx_left
        .withColumn("is_imputed", F.col("close").isNull())
        .withColumn("open",  F.last("open",  ignorenulls=True).over(w_ffill))
        .withColumn("high",  F.last("high",  ignorenulls=True).over(w_ffill))
        .withColumn("low",   F.last("low",   ignorenulls=True).over(w_ffill))
        .withColumn("close", F.last("close", ignorenulls=True).over(w_ffill))
        .withColumn("volume", F.lit(0).cast(LongType()))) 

fx_aligned = fx_filled.join(F.broadcast(kalendarz_gieldy), on="date", how="inner")

display(fx_aligned.orderBy("ticker", "date").limit(10))

In [0]:
KOLUMNY = ["date", "ticker", "open", "high", "low", "close", "volume", "is_imputed", "ingested_at"]
stk_final = stk.withColumn("is_imputed", F.lit(False)).select(*KOLUMNY)
fx_final  = fx_aligned.select(*KOLUMNY)

silver_ffill = stk_final.unionByName(fx_final)
display(silver_ffill.orderBy("ticker", F.col("date").desc()).limit(10))

In [0]:
fx_one  = fx.filter(F.col("ticker") == "EURUSD=X")
stk_one = stk.filter(F.col("ticker") == "^GSPC")

fx_w = fx_one.select(
    "date",
    F.col("open").alias("eurusd_open"),
    F.col("high").alias("eurusd_high"),
    F.col("low").alias("eurusd_low"),
    F.col("close").alias("eurusd_close"),
)

stk_w = stk_one.select(
    "date",
    F.col("open").alias("gspc_open"),
    F.col("high").alias("gspc_high"),
    F.col("low").alias("gspc_low"),
    F.col("close").alias("gspc_close"),
    F.col("volume").alias("gspc_volume"),
)

silver_ij = (stk_w
             .join(F.broadcast(fx_w), on="date", how="inner")
             .orderBy("date"))

display(silver_ij.limit(10))

In [0]:
dni_akcji = stk.filter(F.col("ticker") == "^GSPC").select("date").distinct()
dni_fx    = fx.filter(F.col("ticker") == "EURUSD=X").select("date").distinct()

wspolne = dni_akcji.join(dni_fx, on="date", how="inner")

tylko_akcje = dni_akcji.join(dni_fx, on="date", how="left_anti")

print(dni_akcji.count())
print(dni_fx.count())
print(wspolne.count())
print(tylko_akcje.count())


In [0]:
SILVER_BASE = f"abfss://datalake@{storage_account}.dfs.core.windows.net/silver"

(silver_ffill.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{SILVER_BASE}/market_ohlcv"))

(silver_ij.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{SILVER_BASE}/market_ij"))

In [0]:
silver = spark.read.format("delta").load(f"{SILVER_BASE}/market_ohlcv")

display(silver.groupBy("ticker").agg(
    F.count("*").alias("wiersze"),
    F.sum(F.col("is_imputed").cast("int")).alias("dni_imputowane"),
    F.min("date").alias("od"),
    F.max("date").alias("do"),
))

dup = silver.groupBy("ticker", "date").count().filter("count > 1").count()
assert dup == 0, f"UWAGA: duplikaty klucza (ticker, date): {dup}!"
print(f"Klucz (ticker, date) unikalny OK (duplikatów: {dup})")
